### Imports

In [1]:
import mlflow
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from mlflow.genai.optimize import GepaPromptOptimizer

from mlflow.genai import scorer

load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "prompt_optimization"

### Load the dataset and save 50 samples

#### Uncomment these cells when running for the first time

In [2]:
# Load AG News dataset from Hugging Face as pandas dataframe
from datasets import load_dataset

dataset = load_dataset("ag_news", split="train")
df = dataset.to_pandas()

df["label"] = df["label"].map({0: "World", 1: "Sports", 2: "Business", 3: "Science"})

df = df.sample(frac=1).reset_index(drop=True)

df.head()

d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,text,label
0,The Bush-Kerry Tax Duel It may be of great int...,Business
1,Company Recalls Vioxx Arthritis Drug The compa...,Business
2,"28,000 lives may be about to change ALEXANDRIA...",Business
3,New \$50 Bill Begins Circulating Coming to cas...,Business
4,Microsoft Seals #39;Windows #39; in Server 20...,Science


In [3]:
NUM_SAMPLES = 20
train_data = []
for i in range(NUM_SAMPLES):
    article = df.iloc[i]["text"]
    expected = df.iloc[i]["label"]
    eval_dict = {
        "inputs": {"article": article},
        "expectations": {"expected_response": expected},
    }
    train_data.append(eval_dict)

train_data[0]

{'inputs': {'article': 'The Bush-Kerry Tax Duel It may be of great interest to voters whether Sen. John F. Kerry was or was not a war hero. That certainly seems to be the issue of the day.'},
 'expectations': {'expected_response': 'Business'}}

### Initialize the llm

In [4]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    rate_limiter=rate_limiter,
)

### Register an initial prompt

In [5]:
prompt = """
You are a helpful assistant that can classify news articles into one of the following categories:
- World
- Sports
- Business
- Science
Article: {article}
"""

initial_prompt = mlflow.genai.register_prompt(
    name="news_classifier",
    template=prompt,
)

2025/10/28 13:38:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: news_classifier, version 1


### Create a predict function and test with a sample

In [ ]:
prompt_uri = initial_prompt.uri


def predict_fn(article) -> str:
    # print("Using prompt uri: ", prompt_uri)
    prompt_template = mlflow.genai.load_prompt(prompt_uri).template
    prompt = prompt_template.format(article=article)
    response = llm.invoke(prompt)
    return response.content

In [7]:
predict_fn(train_data[0]["inputs"]["article"])

Using prompt uri:  prompts:/news_classifier/1


"This article doesn't fit neatly into the provided categories. It discusses a political debate between John F. Kerry and George W. Bush, focusing on their tax policies and Kerry's military service. While it touches on political figures, the core subject matter is not international affairs (World), athletic competitions (Sports), financial markets or companies (Business), or technological advancements and discoveries (Science).\n\nTherefore, I cannot classify this article into one of the given categories."

### Create an baseline

In [8]:
@scorer
def exact_match(outputs, expectations):
    expectations = expectations["expected_response"]
    return outputs == expectations


with mlflow.start_run(run_name="optimize-prompt-baseline"):
    results = mlflow.genai.evaluate(
        data=train_data,
        scorers=[exact_match],
        predict_fn=predict_fn,
    )

2025/10/28 13:38:47 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.


Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1


Evaluating:   5%|▌         | 1/20 [Elapsed: 00:14, Remaining: 04:32] 

Using prompt uri:  prompts:/news_classifier/1


Evaluating:  10%|█         | 2/20 [Elapsed: 00:17, Remaining: 02:37] 

...

Evaluating:  45%|████▌     | 9/20 [Elapsed: 01:29, Remaining: 01:49] 

Using prompt uri:  prompts:/news_classifier/1


Evaluating:  50%|█████     | 10/20 [Elapsed: 01:39, Remaining: 01:39] 

Using prompt uri:  prompts:/news_classifier/1


Evaluating: 100%|██████████| 20/20 [Elapsed: 03:19, Remaining: 00:00] 


### Optimize the prompt

In [9]:
result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=train_data,
    prompt_uris=[initial_prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model="openai:/gpt-5-mini"),
    scorers=[exact_match],
)

2025/10/28 13:42:17 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.


Using prompt uri:  prompts:/news_classifier/1


d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'prompt_optimization_train_data'. Exception: 
  return _dataset_source_registry.resolve(
d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Using prompt uri:  prompts:/news_classifier/1
Iteration 0: Base program full valset score: 0.75
Iteration 1: Selected program 

2025/10/28 13:59:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: news_classifier, version 2


Iteration 11: New subsample score 1.0 is not better than old score 2.0, skipping
🏃 View run rebellious-mole-95 at: http://localhost:5000/#/experiments/560692231167391968/runs/06a458b9a3a243f99309d51416a664c9
🧪 View experiment at: http://localhost:5000/#/experiments/560692231167391968


### Run the evaluation with Optimized prompt

In [10]:
prompt_uri = result.optimized_prompts[0].uri
sample_result = predict_fn(train_data[0]["inputs"]["article"])
print(sample_result)

Using prompt uri:  prompts:/news_classifier/2
World


In [11]:
with mlflow.start_run(run_name="optimized-prompt-eval"):
    results = mlflow.genai.evaluate(
        data=train_data,
        scorers=[exact_match],
        predict_fn=predict_fn,
    )

2025/10/28 13:59:46 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/10/28 13:59:46 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.


Using prompt uri:  prompts:/news_classifier/2
Using prompt uri: Using prompt uri:  prompts:/news_classifier/2
 prompts:/news_classifier/2
Using prompt uri:  prompts:/news_classifier/2
Using prompt uri:  prompts:/news_classifier/2
Using prompt uri:  prompts:/news_classifier/2
Using prompt uri:  prompts:/news_classifier/2
Using prompt uri:  prompts:/news_classifier/2
Using prompt uri:  prompts:/news_classifier/2
Using prompt uri:  prompts:/news_classifier/2
Using prompt uri:  prompts:/news_classifier/2


Evaluating:   5%|▌         | 1/20 [Elapsed: 00:12, Remaining: 03:52] 

Using prompt uri:  prompts:/news_classifier/2


Evaluating:  10%|█         | 2/20 [Elapsed: 00:20, Remaining: 03:02] 

...

Evaluating:  45%|████▌     | 9/20 [Elapsed: 01:30, Remaining: 01:50] 

Using prompt uri:  prompts:/news_classifier/2


Evaluating:  50%|█████     | 10/20 [Elapsed: 01:50, Remaining: 01:50] 

Using prompt uri:  prompts:/news_classifier/2


Evaluating: 100%|██████████| 20/20 [Elapsed: 03:22, Remaining: 00:00] 
